# Task 3: U-Net Artery Segmentation (Syntax Score Dataset)

This notebook trains a U-Net model with a ResNet34 backbone using `segmentation-models-pytorch` specifically customized for artery tree branch segmentation using the Syntax Score dataset.

In [ ]:
# Mount Google Drive to access your dataset
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Install required packages
!pip install segmentation-models-pytorch albumentations opencv-python

In [ ]:
import os
import cv2
import random
import numpy as np
from glob import glob
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import albumentations as A
from albumentations.pytorch import ToTensorV2
import segmentation_models_pytorch as smp

In [ ]:
# ----- CONFIGURATION -----
CONFIG = {
    "train_images": "/content/drive/MyDrive/syntax/train/preprocessed",  # preprocessed images folder
    "train_masks":  "/content/drive/MyDrive/syntax/train/mask",          # preprocessed masks folder
    "val_images":   "/content/drive/MyDrive/syntax/val/preprocessed",
    "val_masks":    "/content/drive/MyDrive/syntax/val/mask",
    "img_size":     (512, 512),        # (W, H)
    "batch_size":   8,
    "num_workers":  4,
    "epochs":       60,
    "learning_rate": 1e-4,
    "weight_decay": 1e-5,
    "num_classes":  1,                 # 1 for binary segmentation
    "device":       "cuda" if torch.cuda.is_available() else "cpu",
    "save_dir":     "models/unet",
    "encoder":      "resnet34",
    "encoder_weights": "imagenet",
    "activation":   None,
    "patience":     10,
}

os.makedirs(CONFIG["save_dir"], exist_ok=True)

def seed_everything(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

seed_everything(42)

In [ ]:
class ArteryDataset(Dataset):
    def __init__(self, image_dir, mask_dir, img_size=(512,512), augment=False, num_classes=1):
        self.image_paths = sorted(glob(os.path.join(image_dir, "*")))
        self.mask_paths = []
        for p in self.image_paths:
            fname = os.path.basename(p)
            mp = os.path.join(mask_dir, fname)
            if not os.path.exists(mp):
                mp = None
            self.mask_paths.append(mp)
        self.img_size = img_size
        self.augment = augment
        self.num_classes = num_classes
        self._build_transforms()

    def _build_transforms(self):
        # Strong augmentation for training
        train_transforms = [
            A.Resize(self.img_size[1], self.img_size[0]),
            A.HorizontalFlip(p=0.5),
            A.VerticalFlip(p=0.2),
            A.ShiftScaleRotate(shift_limit=0.0625, scale_limit=0.1, rotate_limit=15, p=0.5, border_mode=cv2.BORDER_REFLECT),
            A.RandomBrightnessContrast(p=0.5),
            A.GaussNoise(p=0.2),
            A.Normalize(mean=(0.0,0.0,0.0), std=(1.0,1.0,1.0)),
            ToTensorV2()
        ]
        valid_transforms = [
            A.Resize(self.img_size[1], self.img_size[0]),
            A.Normalize(mean=(0.0,0.0,0.0), std=(1.0,1.0,1.0)),
            ToTensorV2()
        ]
        self.train_tf = A.Compose(train_transforms)
        self.valid_tf = A.Compose(valid_transforms)

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        img_path = self.image_paths[idx]
        mask_path = self.mask_paths[idx]

        img = cv2.imread(img_path, cv2.IMREAD_COLOR)
        if img is None:
            raise FileNotFoundError(f"{img_path} not found")

        if mask_path is None:
            mask = np.zeros((img.shape[0], img.shape[1]), dtype=np.uint8)
        else:
            mask = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)
            if mask is None:
                mask = np.zeros((img.shape[0], img.shape[1]), dtype=np.uint8)

        if self.num_classes == 1:
            mask = (mask > 127).astype("float32")
        else:
            h, w = mask.shape
            onehot = np.zeros((self.num_classes, h, w), dtype=np.float32)
            for c in range(self.num_classes):
                onehot[c] = (mask == c).astype(np.float32)
            mask = onehot

        if self.num_classes == 1:
            aug = self.train_tf(image=img, mask=mask) if self.augment else self.valid_tf(image=img, mask=mask)
            image = aug["image"]
            mask_t = aug["mask"][None, ...]
            mask_t = mask_t.float()
        else:
            if self.augment:
                image = self.train_tf(image=img)["image"]
            else:
                image = self.valid_tf(image=img)["image"]
            mask_resized = cv2.resize(mask, (self.img_size[0], self.img_size[1]), interpolation=cv2.INTER_NEAREST)
            mask_t = torch.from_numpy(mask_resized).float()
            if mask_t.ndim == 2:
                onehot = torch.zeros((self.num_classes, self.img_size[1], self.img_size[0]), dtype=torch.float32)
                for c in range(self.num_classes):
                    onehot[c] = (mask_t == c).float()
                mask_t = onehot

        return image, mask_t

In [ ]:
# Create loaders
train_dataset = ArteryDataset(
    CONFIG["train_images"], CONFIG["train_masks"], 
    img_size=CONFIG["img_size"], augment=True, num_classes=CONFIG["num_classes"]
)
val_dataset = ArteryDataset(
    CONFIG["val_images"], CONFIG["val_masks"], 
    img_size=CONFIG["img_size"], augment=False, num_classes=CONFIG["num_classes"]
)

train_loader = DataLoader(
    train_dataset, batch_size=CONFIG["batch_size"], shuffle=True, 
    num_workers=CONFIG["num_workers"], pin_memory=True
)
val_loader = DataLoader(
    val_dataset, batch_size=CONFIG["batch_size"], shuffle=False, 
    num_workers=CONFIG["num_workers"], pin_memory=True
)

In [ ]:
# Initialize model
model = smp.Unet(
    encoder_name=CONFIG["encoder"],
    encoder_weights=CONFIG["encoder_weights"],
    in_channels=3,
    classes=CONFIG["num_classes"],
    activation=CONFIG["activation"]
).to(CONFIG["device"])

# Losses
bce_loss = nn.BCEWithLogitsLoss()

def dice_loss_logits(pred, target, eps=1e-6):
    pred = torch.sigmoid(pred)
    num = 2.0 * (pred * target).sum(dim=(1, 2, 3))
    den = pred.sum(dim=(1, 2, 3)) + target.sum(dim=(1, 2, 3))
    loss = 1.0 - (num + eps) / (den + eps)
    return loss.mean()

def get_loss_fn(num_classes=1, bce_weight=1.0, dice_weight=1.0):
    def loss_fn(pred, target):
        return bce_weight * bce_loss(pred, target) + dice_weight * dice_loss_logits(pred, target)
    return loss_fn

criterion = get_loss_fn(num_classes=CONFIG["num_classes"])
optimizer = optim.AdamW(model.parameters(), lr=CONFIG["learning_rate"], weight_decay=CONFIG["weight_decay"])
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="max", factor=0.5, patience=3)

In [ ]:
def iou_score(preds, targets, threshold=0.5, eps=1e-6):
    preds = (torch.sigmoid(preds) > threshold).float()
    intersection = (preds * targets).sum(dim=(1, 2, 3))
    union = preds.sum(dim=(1, 2, 3)) + targets.sum(dim=(1, 2, 3)) - intersection
    iou = (intersection + eps) / (union + eps)
    return iou.mean().item()

# ----- TRAINING & VALIDATION -----
best_iou = 0.0
patience_counter = 0

for epoch in range(CONFIG["epochs"]):
    model.train()
    train_loss = 0.0
    for images, masks in train_loader:
        images, masks = images.to(CONFIG["device"]), masks.to(CONFIG["device"])
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, masks)
        loss.backward()
        optimizer.step()
        train_loss += loss.item() * images.size(0)
        
    train_loss /= len(train_loader.dataset)
    
    # Validation
    model.eval()
    val_loss, val_iou = 0.0, 0.0
    with torch.no_grad():
        for images, masks in val_loader:
            images, masks = images.to(CONFIG["device"]), masks.to(CONFIG["device"])
            outputs = model(images)
            loss = criterion(outputs, masks)
            val_loss += loss.item() * images.size(0)
            val_iou += iou_score(outputs, masks) * images.size(0)
            
    val_loss /= len(val_loader.dataset)
    val_iou /= len(val_loader.dataset)
    
    scheduler.step(val_iou)
    print(f"Epoch {epoch+1}/{CONFIG['epochs']} | Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f} | Val IoU: {val_iou:.4f}")
    
    if val_iou > best_iou:
        best_iou = val_iou
        patience_counter = 0
        torch.save(model.state_dict(), os.path.join(CONFIG["save_dir"], "best_unet_resnet34_syntax.pth"))
        print(f"==> Saved best model with IoU: {best_iou:.4f}")
    else:
        patience_counter += 1
        
    if patience_counter >= CONFIG["patience"]:
        print(f"Early stopping triggered after {epoch+1} epochs.")
        break